# Phase 6 (Abstractive Summarization) — Run #1: Phase-1 **Decoder-only model 02** (hybrid local/global attention + GQA) with **LoRMA**

Config-driven like the Phase-2…5 notebooks: edit **Cell 2** only. This is the first Phase-6 run: Dec-02 + **LoRMA** (the compulsory technique, Bihany, Patel & Modi 2025). It is also a **KD-teacher candidate** (`EXPORT_TEACHER=True`): it exports top-k logits on the gold targets and teacher-generated chunk summaries for run #3.

### What the data audit (Step 0) found, and what it decided here
| Audit finding | Consequence in this notebook |
|---|---|
| 7,030 train / 100 test, no dev split; 27 duplicate documents | Fixed validation split of 200 docs (`split_train`, copied verbatim from the audit); `VAL_CHECKSUM == 5720e140c147` is **asserted** |
| Docs: median 3.8k tokens (p90 8.8k). Summaries: median 739 tokens, 72 % > 512. Only **1 %** of (doc, summary) pairs fit a 1024 context | **Chunk-level summarisation**: every training example is one 512-token chunk → the part of the summary that came from it |
| Summary sentences align well to the document (median F1 0.7) and follow its **order** (77 % monotone) | Targets are built by sentence alignment; at test time the chunk summaries are **concatenated in document order** |
| 512-token chunks: 25 % get no summary text, target median 114 tokens, p90 277 | `CHUNK_TOKENS = 512`, `TGT_MAX = 384` (fits: 1 + tag + 512 + sep + 384 + 1 < 1024); half of the empty chunks are kept with an EOS-only target so the model learns to *skip* |
| Aligned positions are U-shaped (start 11 %, end 14 %) | Documents with more than `max_chunks` chunks keep head + tail + evenly spread middle chunks; a `[part i/n]` tag tells the model where it is |
| Very extractive references (fragment coverage 0.96, density 14.7) | Greedy decoding with **no-repeat-n-gram** + a mild repetition penalty (copying allowed, loops blocked), sentence de-duplication |
| BERTScore (roberta-large) **truncates at ~410 words**; ROUGE-1 peaks at ~1.25× the predicted length | Document-order output (references open with the facts); total length = `1.25 × 4.54·|doc|^0.61` words, clipped to [232, 2002] |
| TF-IDF centroid: R1 0.575 / R2 0.319 / RL 0.301 / BERTScore 0.853 | The bar printed next to every result |

### Model and training
* **Input** (decoder-only): `[BOS] [part i/n] <512 document tokens> \n\nSUMMARY:\n <target> [EOS]`. **Loss only on the target** (prompt labels = −100), i.e. conditional LM: maximise `log p(y | x) = Σ_t log p(y_t | x, y_<t)`.
* **LoRMA+** on every projection of every block (q, k, v, o, gate, up, down): `h = (I + (α/r)·B A)·W₀x`, `W₀` frozen, `B = 0` at init so step 0 is exactly the Phase-1 model (checked). LM head = tied embedding, frozen.
* Causal Phase-1 masks (needed for generation); the local layers keep their 256-token window, the global layers (3, 7, 11) see the whole prompt. Right-padded training batches need no padding mask (a causal query never sees later padding), so the fast fused causal path is used.
* Best epoch by **validation loss** on the VAL chunk pairs (`SELECT_BY`), with a quick ROUGE check on `VAL_GEN_DOCS_EPOCH` documents every epoch.

### Run it
1. `SMOKE_TEST=True` first (~15 min): the loss must fall and the sample outputs must look like legal sentences. The `>> speed` line projects the full run.
2. `SMOKE_TEST=False`: train → decoding sweep on VAL → VAL evaluation (ROUGE + BERTScore) → test submission → teacher export. Paste the **"PASTE TO CLAUDE"** block (Cell 27).

In [1]:
# Cell 0 — Hugging Face login (token from Kaggle secrets), same as the Phase-1 notebooks
import os
try:
    from huggingface_hub import login
    from kaggle_secrets import UserSecretsClient
    _tok = UserSecretsClient().get_secret("HF")
    os.environ["HF_TOKEN"] = _tok
    login(_tok)
    print("HF login ok")
except Exception as e:
    print("HF secret not available:", e)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


HF login ok


In [2]:
# Cell 1 — Imports, paths, device, precision (same conventions as the Phase-1 notebooks)
import os, sys, json, math, time, random, gc, zipfile, types, re, pickle, copy, hashlib, bisect, subprocess
from pathlib import Path
from dataclasses import dataclass, asdict, field
from typing import Optional, Dict, List, Tuple, Any
from collections import Counter, defaultdict

import multiprocessing as mp
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint

IS_KAGGLE = Path("/kaggle/working").exists()
WORK_DIR = Path("/kaggle/working") if IS_KAGGLE else Path("./work")
WORK_DIR.mkdir(parents=True, exist_ok=True)
TOKENIZER_DIR = Path("/kaggle/input/datasets/ascyrax/capstone-tokenizer-large") if IS_KAGGLE else Path("./input")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    cap = torch.cuda.get_device_capability(0)
    AMP_DTYPE = torch.bfloat16 if cap[0] >= 8 else torch.float16     # T4/P100 -> fp16 + GradScaler
    USE_AMP = True
    print("GPU:", torch.cuda.get_device_name(0), "| capability", cap)
else:
    AMP_DTYPE, USE_AMP = torch.float32, False
USE_SCALER = USE_AMP and AMP_DTYPE == torch.float16
print(f"device={device} | AMP dtype={AMP_DTYPE} | grad scaler={USE_SCALER} | torch {torch.__version__}")

SEED = 1337
def seed_everything(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if device.type == "cuda": torch.cuda.manual_seed_all(s)
seed_everything(SEED)
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

GPU: Tesla T4 | capability (7, 5)
device=cuda | AMP dtype=torch.float16 | grad scaler=True | torch 2.11.0+cu128


In [3]:
# Cell 2 — RUN CONFIG  (the only cell you normally edit)

RUN_NAME   = "p6_dec02_lorma"
METHOD     = "lorma"          # Phase-6 list: LoRMA (compulsory). HP["lorma"]["variant"]: "plus" (LoRMA+) | "pi" (LoRMA_pi)
SMOKE_TEST = False            # True: ~150 steps on a small subset, a few VAL documents generated, NO submission / export

# ---- Phase-1 checkpoint (written by caps_decoder_only_architecture_02.ipynb: runs/arch02_hybrid_gqa/best.pt) ----------
CKPT_PATH = Path("/kaggle/input/datasets/ascyrax/caps-dec-02-best/best.pt") if IS_KAGGLE else Path("./input/best.pt")

# ---- Data / split (audit) ------------------------------------------------------------------------
HF_REPO        = "Exploration-Lab/CS-781-Capstone"     # Phase_6_Summ/{Train/train.json, Test/test.json}
LOCAL_DATA_DIR = None
P6             = "Phase_6_Summ"
VAL_N, VAL_SALT, VAL_CHECKSUM_EXPECTED = 200, "p6-val-v1", "5720e140c147"
MIN_SUMM_WORDS, MAX_SUMM_WORDS = 20, 5000       # training pairs only (VAL is kept as in the audit)
MAX_FULL_CHARS = 150_000                        # longer documents (< 1 %) -> first 2/3 + last 1/3 of this many characters

# ---- Chunk pairs ---------------------------------------------------------------------------------
CHUNK_TOKENS = 512           # document tokens per chunk
TGT_MAX      = 384           # target tokens per chunk (audit: p90 277; longer targets are cut at a sentence boundary)
MIN_ALIGN_F1 = 0.2           # weaker alignments are not assigned to a chunk (audit: 1.9 % of summary sentences)
EMPTY_KEEP   = 0.5           # share of chunks WITHOUT aligned summary text kept as EOS-only examples (resampled every epoch)
TAG_TEMPLATE = "[part {i}/{n}] "
SEP_TEXT     = "\n\nSUMMARY:\n"
N_PROC       = 4

# ---- Training ------------------------------------------------------------------------------------
EPOCHS        = 4            # smoke run: ~0.8 s/step on a T4 -> ~25-30 min/epoch on the full data
MICRO_TOKENS  = 12288        # padded tokens per micro-batch (n_seqs x longest); lower it on OOM
STEP_SEQS     = 32           # sequences per optimiser step (gradient accumulation)
WARMUP_FRAC   = 0.03
GRAD_CLIP     = 1.0
GRAD_CKPT     = False
TIME_BUDGET_MIN = 400        # training stops after this (best epoch kept); leaves ~3 h for evaluation, test and export
HP = {"lorma": dict(lr=6e-4, wd=0.0, r=16, alpha=32, variant="plus")}
ADAPTER_TARGETS = ("q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj")
SELECT_BY          = "loss"  # "loss" (VAL teacher-forced loss) | "rouge" (quick ROUGE on VAL_GEN_DOCS_EPOCH docs)
VAL_GEN_DOCS_EPOCH = 40

# ---- Generation ----------------------------------------------------------------------------------
GEN = dict(max_chunks=16,        # audit: 16 x 512 reads 95.7 % of all tokens
           max_new=TGT_MAX, min_new_first=32,   # the first chunk must say something (no empty summaries)
           min_new_all=0,                       # [EOS] banned for this many tokens in EVERY chunk (length control; chosen by the sweep)
           no_repeat_ngram=4, rep_penalty=1.1, batch=64,
           len_factor=1.25, dedup=True)
DECODE_SWEEP = [dict(no_repeat_ngram=4, rep_penalty=1.1, min_new_all=0),  dict(no_repeat_ngram=4, rep_penalty=1.1, min_new_all=32),
                dict(no_repeat_ngram=4, rep_penalty=1.1, min_new_all=64), dict(no_repeat_ngram=3, rep_penalty=1.0, min_new_all=32),
                dict(no_repeat_ngram=0, rep_penalty=1.2, min_new_all=32)]
SWEEP_DOCS   = 60            # VAL docs used to pick the decoding config (by (R1 + RL) / 2); [] / 0 = skip
LEN_MODEL    = dict(a=4.538474, b=0.6144775, lo=232.0, hi=2002.1)       # audit power law (words)
RUN_BERTSCORE, BERTSCORE_MODEL = True, "roberta-large"                   # metric only, never used for training
BASELINES = dict(tfidf=dict(r1=0.5749, r2=0.3186, rL=0.3008, bs=0.8533), lead=dict(r1=0.5222, r2=0.2508, rL=0.2437, bs=0.8321),
                 oracle=dict(r1=0.786, r2=0.6042, rL=0.5341, bs=0.8855))

# ---- KD teacher export (run #3) -------------------------------------------------------------------
EXPORT_TEACHER    = True
TEACHER_TOPK      = 8        # top-k logits per gold target token (+ logsumexp), all TR pairs
TEACHER_GEN_PAIRS = 15000    # teacher-generated summaries for this many TR chunks (sequence-level KD)
TEACHER_GEN_MIN   = 90       # time cap (minutes) for the generation part

# ---- Smoke test ----------------------------------------------------------------------------------
SMOKE_STEPS, SMOKE_TRAIN_DOCS, SMOKE_VAL_DOCS = 150, 400, 12

# ---- derived -------------------------------------------------------------------------------------
assert METHOD in HP and SELECT_BY in ("loss", "rouge")
BODY = CHUNK_TOKENS
RUN_DIR = WORK_DIR / "runs" / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)
print(f"RUN {RUN_NAME} | METHOD={METHOD} {HP[METHOD]} | chunk {CHUNK_TOKENS} -> target <= {TGT_MAX} | empty keep {EMPTY_KEEP} | "
      f"epochs {EPOCHS} | micro {MICRO_TOKENS} tok, step {STEP_SEQS} seqs | smoke={SMOKE_TEST}")

RUN p6_dec02_lorma | METHOD=lorma {'lr': 0.0006, 'wd': 0.0, 'r': 16, 'alpha': 32, 'variant': 'plus'} | chunk 512 -> target <= 384 | empty keep 0.5 | epochs 4 | micro 12288 tok, step 32 seqs | smoke=False


In [4]:
# Cell 3 — The shared byte-level BPE tokenizer (same as every Phase-1 model)
if str(TOKENIZER_DIR) not in sys.path:
    sys.path.append(str(TOKENIZER_DIR))
from bpe_tokenizer import BPETokenizer, load_tokenizer

tokenizer = load_tokenizer(str(TOKENIZER_DIR))
VOCAB_SIZE = len(tokenizer.id_to_bytes)
BOS_ID, EOS_ID, PAD_ID, UNK_ID = tokenizer.bos_id, tokenizer.eos_id, tokenizer.pad_id, tokenizer.unk_id
print("vocab_size:", VOCAB_SIZE, "| BOS", BOS_ID, "EOS", EOS_ID, "PAD", PAD_ID)

_s = "Hon'ble Mr. Justice R. K. Gauba — § 302 IPC, ₹ 5,000 “quoted”."
_ids, _offs = tokenizer.encode(_s, return_offsets=True)
assert tokenizer.decode(_ids) == _s, "tokenizer round-trip failed"
assert all(0 <= a <= b <= len(_s) for a, b in _offs) and _offs[0][0] == 0 and _offs[-1][1] == len(_s)
print("offset sanity ok |", len(_ids), "tokens for", len(_s), "chars")

# Guard against O(L^2) BPE on pathological chunks (long runs of dashes, dots, underscores, spaces) -- as in Phases 3-4.
_MAX_PRETOK_BYTES = 1000
_orig_encode_pretok = tokenizer._encode_pretok_bytes
def _encode_pretok_bytes_capped(bs):
    if len(bs) <= _MAX_PRETOK_BYTES:
        return _orig_encode_pretok(bs)
    out = []
    for i in range(0, len(bs), _MAX_PRETOK_BYTES):
        out.extend(_orig_encode_pretok(bs[i:i + _MAX_PRETOK_BYTES]))
    return out
tokenizer._encode_pretok_bytes = _encode_pretok_bytes_capped
_t0 = time.time(); _ids = tokenizer.encode("-" * 200_000)
assert tokenizer.decode(_ids) == "-" * 200_000
print(f"long-run guard ok: 200k-char run encoded in {time.time() - _t0:.1f}s")

vocab_size: 32000 | BOS 2 EOS 3 PAD 0
offset sanity ok | 28 tokens for 62 chars
long-run guard ok: 200k-char run encoded in 0.1s


## Phase-1 model code — copied **verbatim** from `caps_decoder_only_architecture_02.ipynb` (Cells 4-11)
Do not edit: the checkpoint only loads correctly if these classes are identical.

In [5]:
# Cell 4 — Model config (serialisable; saved inside every checkpoint)
@dataclass
class ModelConfig:
    # --- shape ---
    vocab_size: int = 32000
    n_layer: int = 12
    d_model: int = 512
    n_head: int = 8
    n_kv_head: int = 8                 # < n_head  => GQA (each KV head is shared by n_head//n_kv_head query heads)
    head_dim: int = 64
    ff_mult: float = 8 / 3             # SwiGLU hidden size ~ (8/3)*d_model, rounded up to a multiple of 64
    d_ff: Optional[int] = None         # if set, overrides ff_mult
    max_seq_len: int = 1024

    # --- attention pattern ---
    local_window: int = 0              # 0 => every layer is full causal attention
    global_layers: List[int] = field(default_factory=list)   # 0-indexed layers that stay FULL when local_window > 0

    # --- components ---
    norm_type: str = "rmsnorm"         # "rmsnorm" | "layernorm"
    norm_eps: float = 1e-5
    pos_type: str = "rope"             # "rope" | "none"
    rope_theta: float = 10000.0
    mlp_type: str = "swiglu"           # "swiglu" | "gelu"
    mixer_type: str = "attention"      # token-mixer registry key (swap in other mixers later)
    channel_mixer_type: str = "mlp"    # channel-mixer registry key

    # --- misc ---
    bias: bool = False
    attn_dropout: float = 0.0
    resid_dropout: float = 0.0
    embd_dropout: float = 0.0
    tie_embeddings: bool = True
    init_std: float = 0.02
    use_sdpa: bool = True

    def __post_init__(self):
        assert self.d_model == self.n_head * self.head_dim, "d_model must equal n_head*head_dim"
        assert self.n_head % self.n_kv_head == 0, "n_head must be divisible by n_kv_head"
        assert self.pos_type in ("rope", "none")
        if self.d_ff is None:
            raw = int(self.ff_mult * self.d_model)
            self.d_ff = 64 * ((raw + 63) // 64)
        if self.local_window > 0:
            assert all(0 <= i < self.n_layer for i in self.global_layers)

    def is_global(self, layer_idx: int) -> bool:
        return self.local_window <= 0 or layer_idx in self.global_layers

    def window_for(self, layer_idx: int) -> Optional[int]:
        return None if self.is_global(layer_idx) else self.local_window


def preset_baseline_rope(vocab_size=None, max_seq_len=1024):
    """Model 01's shape (MHA 8/8, d_ff 1408, all layers full) but with RoPE really applied.
    Use this to train a clean controlled baseline for the ablation if you have GPU time."""
    return ModelConfig(vocab_size=vocab_size or VOCAB_SIZE, max_seq_len=max_seq_len)


def preset_hybrid_gqa(vocab_size=None, max_seq_len=1024):
    """Architecture 02: GQA (2 KV heads) + sliding window 256 on 9 of 12 layers + wider FFN to keep params equal."""
    return ModelConfig(vocab_size=vocab_size or VOCAB_SIZE, max_seq_len=max_seq_len,
                       n_kv_head=2, d_ff=1664, local_window=256, global_layers=[3, 7, 11])

print(json.dumps(asdict(preset_hybrid_gqa()), indent=1))

{
 "vocab_size": 32000,
 "n_layer": 12,
 "d_model": 512,
 "n_head": 8,
 "n_kv_head": 2,
 "head_dim": 64,
 "ff_mult": 2.6666666666666665,
 "d_ff": 1664,
 "max_seq_len": 1024,
 "local_window": 256,
 "global_layers": [
  3,
  7,
  11
 ],
 "norm_type": "rmsnorm",
 "norm_eps": 1e-05,
 "pos_type": "rope",
 "rope_theta": 10000.0,
 "mlp_type": "swiglu",
 "mixer_type": "attention",
 "channel_mixer_type": "mlp",
 "bias": false,
 "attn_dropout": 0.0,
 "resid_dropout": 0.0,
 "embd_dropout": 0.0,
 "tie_embeddings": true,
 "init_std": 0.02,
 "use_sdpa": true
}


In [6]:
# Cell 5 — Normalisation layers
class LayerNorm(nn.Module):
    def __init__(self, d: int, eps: float = 1e-5):
        super().__init__()
        self.g = nn.Parameter(torch.ones(d)); self.b = nn.Parameter(torch.zeros(d)); self.eps = eps
    def forward(self, x):
        mean = x.mean(-1, keepdim=True)
        var = x.var(-1, keepdim=True, unbiased=False)
        return (x - mean) / torch.sqrt(var + self.eps) * self.g + self.b


class RMSNorm(nn.Module):
    """RMSNorm: scale by 1/rms(x) only (no mean centering, no bias). Computed in fp32 for stability."""
    def __init__(self, d: int, eps: float = 1e-5):
        super().__init__()
        self.g = nn.Parameter(torch.ones(d)); self.eps = eps
    def forward(self, x):
        rms = torch.sqrt(x.float().pow(2).mean(-1, keepdim=True) + self.eps).to(x.dtype)
        return (x / rms) * self.g


def build_norm(cfg: ModelConfig):
    if cfg.norm_type == "layernorm": return LayerNorm(cfg.d_model, cfg.norm_eps)
    if cfg.norm_type == "rmsnorm":   return RMSNorm(cfg.d_model, cfg.norm_eps)
    raise ValueError(cfg.norm_type)

In [7]:
# Cell 6 — Rotary position embeddings (RoPE)
class RoPE(nn.Module):
    """Rotates each (even, odd) pair of a q/k vector by an angle proportional to its position.
    Then q_i . k_j depends only on the RELATIVE offset i-j. Tables are precomputed; math runs in fp32."""
    def __init__(self, head_dim: int, max_seq_len: int, theta: float = 10000.0):
        super().__init__()
        assert head_dim % 2 == 0
        inv_freq = 1.0 / (theta ** (torch.arange(0, head_dim, 2).float() / head_dim))
        freqs = torch.outer(torch.arange(max_seq_len).float(), inv_freq)       # [max_seq_len, head_dim/2]
        self.register_buffer("cos", freqs.cos(), persistent=False)
        self.register_buffer("sin", freqs.sin(), persistent=False)

    @staticmethod
    def _rotate(x, cos, sin):                      # x: [B,H,T,D]; cos,sin: [*,1,T,D/2]
        x1, x2 = x[..., 0::2], x[..., 1::2]
        return torch.stack([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1).flatten(-2)

    def forward(self, q, k, position_ids):
        """q: [B,Hq,T,D], k: [B,Hkv,T,D], position_ids: [B,T] (absolute positions of these T tokens)."""
        cos = self.cos[position_ids].unsqueeze(1).float()   # [B,1,T,D/2]
        sin = self.sin[position_ids].unsqueeze(1).float()
        dt = q.dtype
        return (self._rotate(q.float(), cos, sin).to(dt),
                self._rotate(k.float(), cos, sin).to(dt))

In [8]:
# Cell 7 — Attention: GQA + optional sliding window + KV cache
_MASK_CACHE = {}

def causal_window_mask(T: int, S: int, window: Optional[int], dev) -> torch.Tensor:
    """Bool mask [1,1,T,S], True = query may attend to key.
    The T queries are the LAST T positions of the S keys (so it also works with a KV cache).
    Query at absolute position p attends keys j with  j <= p  and  (window is None or p - j < window),
    i.e. itself + the previous (window-1) tokens."""
    key = (T, S, window, str(dev))
    m = _MASK_CACHE.get(key)
    if m is None:
        qpos = torch.arange(S - T, S, device=dev).view(T, 1)
        kpos = torch.arange(S, device=dev).view(1, S)
        allowed = kpos <= qpos
        if window is not None:
            allowed = allowed & ((qpos - kpos) < window)
        m = allowed.view(1, 1, T, S)
        if T > 1:                       # cache the big square-ish masks only
            _MASK_CACHE[key] = m
    return m


class Attention(nn.Module):
    """Causal multi-head self-attention.
    - GQA: n_kv_head K/V heads, each shared by n_head // n_kv_head query heads (KV cache shrinks by that factor).
    - Local layers: each token sees only the last `window` tokens. Global layers: window=None (full causal).
    - KV cache stores the (already RoPE-rotated) K and V."""
    def __init__(self, cfg: ModelConfig, layer_idx: int):
        super().__init__()
        self.cfg = cfg
        self.layer_idx = layer_idx
        self.window = cfg.window_for(layer_idx)
        self.n_head, self.n_kv_head, self.head_dim = cfg.n_head, cfg.n_kv_head, cfg.head_dim
        self.scale = 1.0 / math.sqrt(self.head_dim)
        self.q_proj = nn.Linear(cfg.d_model, cfg.n_head * cfg.head_dim, bias=cfg.bias)
        self.k_proj = nn.Linear(cfg.d_model, cfg.n_kv_head * cfg.head_dim, bias=cfg.bias)
        self.v_proj = nn.Linear(cfg.d_model, cfg.n_kv_head * cfg.head_dim, bias=cfg.bias)
        self.o_proj = nn.Linear(cfg.n_head * cfg.head_dim, cfg.d_model, bias=cfg.bias)
        self.attn_dropout = nn.Dropout(cfg.attn_dropout)
        self.resid_dropout = nn.Dropout(cfg.resid_dropout)

    def _split(self, x, n):
        B, T, _ = x.shape
        return x.view(B, T, n, self.head_dim).transpose(1, 2)            # [B,n,T,D]

    def forward(self, x, rope=None, position_ids=None, attention_mask=None, kv_cache=None):
        """x: [B,T,d]; position_ids: [B,T]; attention_mask: [B,S] (1 keep / 0 pad, S = cache + T);
        kv_cache: (k_prev, v_prev) each [B,n_kv_head,S_prev,D]. Returns (out, new_cache)."""
        B, T, _ = x.shape
        q = self._split(self.q_proj(x), self.n_head)
        k = self._split(self.k_proj(x), self.n_kv_head)
        v = self._split(self.v_proj(x), self.n_kv_head)

        if rope is not None:                                              # RoPE on the NEW q/k only
            q, k = rope(q, k, position_ids)

        if kv_cache is not None:
            k = torch.cat([kv_cache[0], k], dim=2)
            v = torch.cat([kv_cache[1], v], dim=2)
        new_cache = (k, v)
        S = k.shape[2]

        if self.n_kv_head != self.n_head:                                 # GQA: share each K/V head
            rep = self.n_head // self.n_kv_head
            k = k.repeat_interleave(rep, dim=1)
            v = v.repeat_interleave(rep, dim=1)

        window_binds = self.window is not None and S > self.window
        need_mask = (attention_mask is not None) or window_binds or (T > 1 and S != T)

        allowed = None
        if need_mask:
            allowed = causal_window_mask(T, S, self.window, x.device)     # [1,1,T,S]
            if attention_mask is not None:
                allowed = allowed & attention_mask[:, None, None, :].bool()
                # a left-padding query would see nothing (-> NaN). Let every query see itself (harmless for real tokens).
                qpos = torch.arange(S - T, S, device=x.device).view(T, 1)
                kpos = torch.arange(S, device=x.device).view(1, S)
                allowed = allowed | (kpos == qpos).view(1, 1, T, S)

        p = self.attn_dropout.p if self.training else 0.0
        if self.cfg.use_sdpa:
            if need_mask:
                out = F.scaled_dot_product_attention(q, k, v, attn_mask=allowed, dropout_p=p, is_causal=False)
            else:                                                         # plain causal (global layer, training)
                out = F.scaled_dot_product_attention(q, k, v, dropout_p=p, is_causal=(T > 1))
        else:                                                             # manual reference path (used in tests)
            if allowed is None:
                allowed = causal_window_mask(T, S, None, x.device)
            scores = (q @ k.transpose(-2, -1)) * self.scale
            scores = scores.masked_fill(~allowed, float("-inf"))
            probs = self.attn_dropout(F.softmax(scores.float(), dim=-1).to(q.dtype))
            out = probs @ v

        out = out.transpose(1, 2).contiguous().view(B, T, -1)
        return self.resid_dropout(self.o_proj(out)), new_cache

In [9]:
# Cell 8 — Feed-forward blocks
class SwiGLU(nn.Module):
    """down( silu(gate(x)) * up(x) )"""
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.gate_proj = nn.Linear(cfg.d_model, cfg.d_ff, bias=cfg.bias)
        self.up_proj = nn.Linear(cfg.d_model, cfg.d_ff, bias=cfg.bias)
        self.down_proj = nn.Linear(cfg.d_ff, cfg.d_model, bias=cfg.bias)
        self.drop = nn.Dropout(cfg.resid_dropout)
    def forward(self, x):
        return self.drop(self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x)))


class GELUMLP(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.fc1 = nn.Linear(cfg.d_model, cfg.d_ff, bias=cfg.bias)
        self.fc2 = nn.Linear(cfg.d_ff, cfg.d_model, bias=cfg.bias)
        self.drop = nn.Dropout(cfg.resid_dropout)
    def forward(self, x):
        return self.drop(self.fc2(F.gelu(self.fc1(x))))

In [10]:
# Cell 9 — Registries (plug in new token mixers / channel mixers later) + transformer block
TOKEN_MIXERS, CHANNEL_MIXERS = {}, {}

def register_token_mixer(name):
    def deco(fn): TOKEN_MIXERS[name] = fn; return fn
    return deco

def register_channel_mixer(name):
    def deco(fn): CHANNEL_MIXERS[name] = fn; return fn
    return deco

@register_token_mixer("attention")
def _build_attention(cfg, layer_idx):
    return Attention(cfg, layer_idx)

@register_channel_mixer("mlp")
def _build_mlp(cfg):
    if cfg.mlp_type == "swiglu": return SwiGLU(cfg)
    if cfg.mlp_type == "gelu":   return GELUMLP(cfg)
    raise ValueError(cfg.mlp_type)


class Block(nn.Module):
    """Pre-norm block:  x = x + mixer(norm(x));  x = x + mlp(norm(x))."""
    def __init__(self, cfg: ModelConfig, layer_idx: int):
        super().__init__()
        self.norm1, self.norm2 = build_norm(cfg), build_norm(cfg)
        self.mixer = TOKEN_MIXERS[cfg.mixer_type](cfg, layer_idx)
        self.cmix = CHANNEL_MIXERS[cfg.channel_mixer_type](cfg)
    def forward(self, x, **kw):
        h, new_cache = self.mixer(self.norm1(x), **kw)
        x = x + h
        x = x + self.cmix(self.norm2(x))
        return x, new_cache

In [11]:
# Cell 10 — Backbone, LM head, weight init
class Backbone(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.emb_drop = nn.Dropout(cfg.embd_dropout)
        self.rope = RoPE(cfg.head_dim, cfg.max_seq_len, cfg.rope_theta) if cfg.pos_type == "rope" else None
        self.blocks = nn.ModuleList([Block(cfg, i) for i in range(cfg.n_layer)])
        self.norm_f = build_norm(cfg)

    def forward(self, input_ids, attention_mask=None, position_ids=None, kv_caches=None):
        B, T = input_ids.shape
        if position_ids is None:                       # default: continue after whatever is in the cache
            past = kv_caches[0][0].shape[2] if kv_caches is not None else 0
            position_ids = (past + torch.arange(T, device=input_ids.device)).unsqueeze(0).expand(B, T)
        x = self.emb_drop(self.tok_emb(input_ids))
        new_caches = []
        for i, blk in enumerate(self.blocks):
            x, nc = blk(x, rope=self.rope, position_ids=position_ids, attention_mask=attention_mask,
                        kv_cache=kv_caches[i] if kv_caches is not None else None)
            new_caches.append(nc)
        return self.norm_f(x), new_caches


class LegalLM(nn.Module):
    """Backbone + LM head tied to the token embedding."""
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.cfg = cfg
        self.backbone = Backbone(cfg)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        if cfg.tie_embeddings:
            self.lm_head.weight = self.backbone.tok_emb.weight

    def forward(self, input_ids, targets=None, attention_mask=None, position_ids=None, kv_caches=None):
        h, new_caches = self.backbone(input_ids, attention_mask=attention_mask,
                                      position_ids=position_ids, kv_caches=kv_caches)
        logits = self.lm_head(h)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.float().view(-1, logits.size(-1)), targets.view(-1), ignore_index=-100)
        return logits, loss, new_caches


def init_weights(model: LegalLM, cfg: ModelConfig):
    """N(0, init_std) everywhere; residual-branch output projections scaled by 1/sqrt(2*n_layer)."""
    for m in model.modules():
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, 0.0, cfg.init_std)
            if m.bias is not None: nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, 0.0, cfg.init_std)
    scale = 1.0 / math.sqrt(2 * cfg.n_layer)
    with torch.no_grad():
        for blk in model.backbone.blocks:
            if hasattr(blk.mixer, "o_proj"):  blk.mixer.o_proj.weight.mul_(scale)
            if hasattr(blk.cmix, "down_proj"): blk.cmix.down_proj.weight.mul_(scale)
            if hasattr(blk.cmix, "fc2"):       blk.cmix.fc2.weight.mul_(scale)

In [12]:
# Cell 11 — Utilities: parameter count, optimizer groups (FIXED), save / load
def count_params(model: nn.Module) -> Dict[str, int]:
    """Tied embedding counted once (model.parameters() de-duplicates shared tensors)."""
    total = sum(p.numel() for p in model.parameters())
    emb = model.backbone.tok_emb.weight.numel()
    return {"total": total, "embedding": emb, "non_embedding": total - emb}


def build_param_groups(model: nn.Module, weight_decay: float):
    """Weight decay on 2-D weight matrices only (attention/MLP projections).
    No decay on norm gains/biases (1-D) and on the embedding.
    (Model 01 matched '.b' in names, which also matched 'backbone.blocks...' -> decay was never applied.)"""
    decay, no_decay = [], []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if p.ndim >= 2 and "tok_emb" not in name:
            decay.append(p)
        else:
            no_decay.append(p)
    return [{"params": decay, "weight_decay": weight_decay},
            {"params": no_decay, "weight_decay": 0.0}]


def save_model(path, model, cfg, extra=None):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(".tmp")
    torch.save({"model": model.state_dict(), "config": asdict(cfg), "extra": extra or {}}, tmp)
    os.replace(tmp, path)


def load_model(path, map_location="cpu"):
    ckpt = torch.load(path, map_location=map_location)
    cfg = ModelConfig(**ckpt["config"])
    model = LegalLM(cfg)
    model.load_state_dict(ckpt["model"])
    return model, cfg, ckpt.get("extra", {})

## Load the Phase-1 checkpoint

In [13]:
# Cell 12 — Load the Phase-1 checkpoint and sanity-check it
_ck = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)
lm_cfg = ModelConfig(**_ck["config"])
lm = LegalLM(lm_cfg)
lm.load_state_dict(_ck["model"])
lm_extra = _ck.get("extra", {})
del _ck
lm = lm.to(device).eval()
pc = count_params(lm)
print(f"Phase-1 model | pos_type={lm_cfg.pos_type} | layers={lm_cfg.n_layer} d_model={lm_cfg.d_model} | "
      f"heads {lm_cfg.n_head}:{lm_cfg.n_kv_head} | d_ff={lm_cfg.d_ff} | local_window={lm_cfg.local_window} "
      f"global_layers={lm_cfg.global_layers} | max_seq_len {lm_cfg.max_seq_len} | params {pc['total']/1e6:.2f}M")
print("checkpoint extra: step", lm_extra.get("step"), "| val_loss", lm_extra.get("val_loss"))
_sent = "The appeal is allowed and the impugned judgment of the High Court is set aside."
_ids = [BOS_ID] + tokenizer.encode(_sent)
_x = torch.tensor([_ids[:-1]], device=device); _y = torch.tensor([_ids[1:]], device=device)
with torch.no_grad(), torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=USE_AMP):
    _, _l, _ = lm(_x, targets=_y)
print(f"LM loss on a legal sentence: {_l.item():.3f}  (ln V = {math.log(VOCAB_SIZE):.2f})")
assert _l.item() < 9.0, "checkpoint looks untrained / mis-loaded"
assert VOCAB_SIZE < 65536, "token ids are stored as uint16"

Phase-1 model | pos_type=rope | layers=12 d_model=512 | heads 8:2 | d_ff=1664 | local_window=256 global_layers=[3, 7, 11] | max_seq_len 1024 | params 54.93M
checkpoint extra: step 12749 | val_loss 1.9441224113106728
LM loss on a legal sentence: 1.888  (ln V = 10.37)


## Phase-6 data, the fixed validation split, cleaning
`split_train` and `_h` are copied **verbatim** from the audit (Cell 5); the checksum assert guarantees that every Phase-6 run is scored on the same 200 documents. Cleaning (too-short / too-long summaries, summaries longer than their document) is applied to the **training** part only.

In [14]:
# Cell 13 — Load train / test, split, clean
from huggingface_hub import hf_hub_download
from tqdm.auto import tqdm

def fetch(rel):
    if LOCAL_DATA_DIR is not None and (Path(LOCAL_DATA_DIR) / rel).exists():
        return Path(LOCAL_DATA_DIR) / rel
    try:
        return Path(hf_hub_download(repo_id=HF_REPO, repo_type="dataset", filename=rel, token=os.environ.get("HF_TOKEN")))
    except Exception as ex:
        raise RuntimeError(f"could not download '{rel}' from {HF_REPO} ({type(ex).__name__}: {ex}). "
                           f"Check the HF login (Cell 0) / internet, or download {P6} and set LOCAL_DATA_DIR.") from ex

ID_KEYS  = ("id", "doc_id", "document_id", "case_id")
DOC_KEYS = ("document", "text", "judgment", "judgement", "source", "case_text", "content")
SUM_KEYS = ("summary", "reference_summary", "abstract", "target", "headnote")
def _first(e, keys, default=None):
    for k in keys:
        if k in e: return e[k]
    return default
def _as_text(v): return "\n".join(str(t) for t in v) if isinstance(v, list) else v

def _read_json_any(path):
    txt = Path(path).read_text(encoding="utf-8")
    try: return json.loads(txt)
    except json.JSONDecodeError: return [json.loads(l) for l in txt.splitlines() if l.strip()]

def parse_docs(raw, split, has_sum):
    if isinstance(raw, dict):
        if isinstance(raw.get("data"), list): items = raw["data"]
        else:
            items = []
            for k, v in raw.items():
                v = dict(v) if isinstance(v, dict) else {"document": v}
                v.setdefault("id", k); items.append(v)
    else:
        items = list(raw)
    docs = []
    for i, e in enumerate(items):
        rid = _first(e, ID_KEYS, i)
        doc = _as_text(_first(e, DOC_KEYS))
        if not isinstance(doc, str): raise ValueError(f"{split} item {rid}: no document field. Keys: {list(e)}")
        summ = _as_text(_first(e, SUM_KEYS)) if has_sum else None
        if has_sum and not isinstance(summ, str): raise ValueError(f"{split} item {rid}: no summary field. Keys: {list(e)}")
        docs.append(dict(uid=f"{split}::{i}", id=rid, doc=doc, summ=summ))
    return docs

def load_split(rel, split, has_sum):
    raw = _read_json_any(fetch(rel))
    print(f"  {rel}: {type(raw).__name__} of {len(raw)}")
    return parse_docs(raw, split, has_sum)

TRAIN = load_split(f"{P6}/Train/train.json", "train", True)
TEST_ALL = load_split(f"{P6}/Test/test.json", "test", False)

# ---- verbatim from the audit (Cell 5) ----
def _norm(t): return re.sub(r"\s+", " ", t).strip().lower()
def _h(t):    return hashlib.md5(_norm(t).encode("utf-8")).hexdigest()
def split_train(train, val_n=VAL_N, salt=VAL_SALT):
    # Whole duplicate-document groups go to one side; group order = md5(salt :: normalised-text hash).
    groups = defaultdict(list)
    for d in train: groups[_h(d["doc"])].append(d)
    order = sorted(groups, key=lambda g: hashlib.md5(f"{salt}::{g}".encode()).hexdigest())
    val, val_keys = [], set()
    for g in order:
        if len(val) >= val_n: break
        val.extend(groups[g]); val_keys.add(g)
    tr = [d for d in train if _h(d["doc"]) not in val_keys]
    checksum = hashlib.md5("|".join(sorted(str(d["id"]) for d in val)).encode()).hexdigest()[:12]
    return tr, val, checksum
# ------------------------------------------

TR_ALL, VAL_ALL, _cs = split_train(TRAIN)
print(f"split: TR {len(TR_ALL)} | VAL {len(VAL_ALL)} | checksum {_cs}")
assert _cs == VAL_CHECKSUM_EXPECTED, f"VAL split differs from the audit ({_cs} != {VAL_CHECKSUM_EXPECTED})"

def _clean(docs):
    keep, drop = [], Counter()
    seen = set()
    for d in docs:
        sw, dw = len(d["summ"].split()), len(d["doc"].split())
        key = (_h(d["doc"]), _h(d["summ"]))
        if sw < MIN_SUMM_WORDS: drop["short summary"] += 1; continue
        if sw > MAX_SUMM_WORDS: drop["very long summary"] += 1; continue
        if sw > dw: drop["summary longer than document"] += 1; continue
        if key in seen: drop["exact duplicate pair"] += 1; continue
        seen.add(key); keep.append(d)
    return keep, dict(drop)
TR_ALL, _dropped = _clean(TR_ALL)
print(f"training docs after cleaning: {len(TR_ALL)} | dropped {_dropped}")

Phase_6_Summ/Train/train.json: reconstructing file:   0%|          |  0.00B /  211MB            

Phase_6_Summ/Train/train.json: downloading bytes:           |  0.00B            

  Phase_6_Summ/Train/train.json: list of 7030


test.json:   0%|          | 0.00/2.73M [00:00<?, ?B/s]

  Phase_6_Summ/Test/test.json: list of 100
split: TR 6830 | VAL 200 | checksum 5720e140c147
training docs after cleaning: 6759 | dropped {'exact duplicate pair': 25, 'very long summary': 43, 'summary longer than document': 1, 'short summary': 2}


## Sentence alignment → chunk-level training pairs
`split_sentences`, `content` and `align_doc` are the audit's (Cell 8), unchanged. For every training document: tokenise it **with character offsets**, cut it into 512-token chunks, align each summary sentence to its best document sentence, and give the sentence to the chunk containing that source sentence. A chunk's target is its sentences **in summary order**, cut at a sentence boundary to `TGT_MAX` tokens (a single over-long sentence is cut and gets no `[EOS]`, so the model never learns to stop mid-sentence). Tokenisation and alignment run in `N_PROC` processes and are cached.

In [15]:
# Cell 14 — Sentence splitter + aligner (verbatim from the audit, Cell 8)
_WRE = re.compile(r"[a-z0-9]+")
def words(t): return _WRE.findall(t.lower())
_ABBR = (r"mr|mrs|ms|dr|hon'ble|honble|no|nos|sr|jr|ltd|pvt|co|corp|inc|vs|v|viz|i\.e|e\.g|etc|cf|art|arts|sec|secs|s|ss|cl|"
         r"para|paras|p|pp|vol|ors|anr|govt|st|rs|prof|smt|shri|sri|ch|app|crl|cr|civ|j|jj|cj|mad|bom|cal|del|all|scc|scr|air|"
         r"u/s|r/w|w\.e\.f|addl|asstt|dist|distt|dept|reg|regd|ft|sq|nos|approx|ibid|supra|id|cri|lj|cri\.l\.j|misc|wp|slp")
_ABBR_END = re.compile(r"(?:\b(?:" + _ABBR + r")|\b[A-Za-z])\.$|^\s*\(?\d{1,3}[.)]$", re.I)
_SPLIT = re.compile(r"(?<=[.?!])[\"')\]]*\s+(?=[\"'(\[]?[A-Z0-9])|\n\s*\n|\n(?=\s*(?:\d{1,3}[.)]\s|\([a-z0-9]{1,4}\)\s))")

def split_sentences(text):
    # Returns [(char_start, sentence)]. Pieces ending in an abbreviation / initial / short number are merged forward.
    out, start = [], 0
    pieces = []
    for m in _SPLIT.finditer(text):
        pieces.append((start, text[start:m.start()])); start = m.end()
    pieces.append((start, text[start:]))
    buf_s, buf = None, ""
    for s, p in pieces:
        if buf_s is None: buf_s, buf = s, p
        else: buf = buf + " " + p
        if not _ABBR_END.search(buf.rstrip()):
            if buf.strip(): out.append((buf_s, buf.strip()))
            buf_s, buf = None, ""
    if buf_s is not None and buf.strip(): out.append((buf_s, buf.strip()))
    return out

_STOP = set(("the a an of to in and or for on by with at from as is was were be been are that this these those it its which who "
             "whom whose has have had not no but if than then so such any all also there their they he she his her him we our us you "
             "your i me my shall may can could would should will said upon into under over between after before against within "
             "without about being do does did same other further thereof therein herein hereby whereas").split())
def content(t): return {w for w in _WRE.findall(t.lower()) if w not in _STOP and len(w) > 1}

def align_doc(doc, summ):
    ds, ss = split_sentences(doc), split_sentences(summ)
    dsets = [content(t) for _, t in ds]
    inv = defaultdict(list)
    for j, st in enumerate(dsets):
        for w in st: inv[w].append(j)
    res = []
    for k, (_, t) in enumerate(ss):
        q = content(t)
        if len(q) < 2: continue
        hits = Counter()
        for w in q:
            for j in inv.get(w, ()): hits[j] += 1
        best_j, best_f = -1, 0.0
        for j, c in hits.items():
            f = 2 * c / (len(q) + len(dsets[j]))
            if f > best_f: best_j, best_f = j, f
        if best_j >= 0:
            res.append(dict(k=k, j=best_j, f=best_f, pos=ds[best_j][0] / max(len(doc), 1), char=ds[best_j][0], words=len(t.split())))
    return ds, ss, res

In [16]:
# Cell 15 — Tokenise documents into chunks (+ chunk targets for documents with a summary); cache; prompt pieces
def cut_text(t):
    if len(t) <= MAX_FULL_CHARS: return t
    h = 2 * MAX_FULL_CHARS // 3
    return t[:h] + "\n" + t[-(MAX_FULL_CHARS - h):]

def _fit_target(sents):
    # sentences (summary order) -> (token ids <= TGT_MAX, truncated?) ; whole sentences while they fit
    chosen, ids = [], []
    for s in sents:
        cand = tokenizer.encode(" ".join(chosen + [s]))
        if len(cand) <= TGT_MAX: chosen.append(s); ids = cand
        else: break
    if chosen: return ids, False
    return tokenizer.encode(sents[0])[:TGT_MAX], True

_DOCS_W, _WITH_TGT = None, False
def _work(i):
    d = _DOCS_W[i]
    text = cut_text(d["doc"])
    ids, offs = tokenizer.encode(text, return_offsets=True)
    ids = list(ids)
    if not ids: ids, offs = [EOS_ID], [(0, 0)]
    starts = list(range(0, len(ids), BODY))
    cchar = [offs[s][0] for s in starts]
    chunks = [np.asarray(ids[s:s + BODY], dtype=np.uint16) for s in starts]
    tgts, trunc = None, None
    if _WITH_TGT and d["summ"] is not None:
        _, ss, res = align_doc(text, d["summ"])
        per = defaultdict(set)
        for r in res:
            if r["f"] >= MIN_ALIGN_F1:
                per[max(0, bisect.bisect_right(cchar, r["char"]) - 1)].add(r["k"])
        tgts, trunc = [], []
        for j in range(len(starts)):
            if not per.get(j): tgts.append(None); trunc.append(False); continue
            t_ids, tr = _fit_target([ss[k][1] for k in sorted(per[j])])
            tgts.append(np.asarray(t_ids, dtype=np.uint16)); trunc.append(tr)
    return dict(chunks=chunks, tgts=tgts, trunc=trunc, ntok=len(ids), words=len(d["doc"].split()))

def prepare_docs(docs, cache_name, with_targets):
    global _DOCS_W, _WITH_TGT
    cdir = WORK_DIR / "cache"; cdir.mkdir(exist_ok=True)
    cp = cdir / f"p6_{cache_name}_v{VOCAB_SIZE}_c{CHUNK_TOKENS}_t{TGT_MAX}_a{MIN_ALIGN_F1}_m{MAX_FULL_CHARS}_{int(with_targets)}.pkl"
    cache = pickle.load(open(cp, "rb")) if cp.exists() else {}
    key = lambda d: (d["uid"], len(d["doc"]), len(d["summ"] or ""))
    todo = [i for i, d in enumerate(docs) if cache.get(d["uid"], (None,))[0] != key(d)]
    if todo:
        _DOCS_W, _WITH_TGT = docs, with_targets
        order = sorted(todo, key=lambda i: -len(docs[i]["doc"]))
        res = None
        if N_PROC > 1:
            try:
                with mp.get_context("fork").Pool(N_PROC) as pool:
                    res = list(tqdm(pool.imap(_work, order, chunksize=8), total=len(order), desc=f"prepare {cache_name}", leave=False))
            except Exception as ex:
                print("parallel preparation failed (", type(ex).__name__, ex, ") -> serial"); res = None
        if res is None:
            res = [_work(i) for i in tqdm(order, desc=f"prepare {cache_name}", leave=False)]
        for i, r in zip(order, res): cache[docs[i]["uid"]] = (key(docs[i]), r)
        pickle.dump(cache, open(cp, "wb"))
    for d in docs: d.update(cache[d["uid"]][1])
    return docs

SEP_IDS = tokenizer.encode(SEP_TEXT)
_TAG_CACHE = {}
def tag_ids(j, n):
    k = (j, n)
    if k not in _TAG_CACHE: _TAG_CACHE[k] = tokenizer.encode(TAG_TEMPLATE.format(i=j + 1, n=n))
    return _TAG_CACHE[k]
def prompt_ids(d, j):
    return [BOS_ID] + tag_ids(j, len(d["chunks"])) + d["chunks"][j].astype(np.int64).tolist() + SEP_IDS
MAX_PROMPT = 1 + len(tag_ids(998, 999)) + BODY + len(SEP_IDS)
assert MAX_PROMPT + TGT_MAX + 1 <= lm_cfg.max_seq_len, f"prompt {MAX_PROMPT} + target {TGT_MAX} exceeds the RoPE table {lm_cfg.max_seq_len}"
print(f"prompt <= {MAX_PROMPT} tokens + target <= {TGT_MAX} + EOS -> sequences <= {MAX_PROMPT + TGT_MAX + 1} (context {lm_cfg.max_seq_len})")

def pair_stats(docs, title):
    n_ch = np.asarray([len(d["chunks"]) for d in docs])
    t = [len(x) for d in docs for x in d["tgts"] if x is not None]
    emp = np.mean([x is None for d in docs for x in d["tgts"]])
    tr = np.mean([f for d in docs for x, f in zip(d["tgts"], d["trunc"]) if x is not None])
    print(f"{title}: {len(docs)} docs | chunks/doc mean {n_ch.mean():.1f} median {np.median(n_ch):.0f} | chunk pairs with a target {len(t)} "
          f"| empty chunks {100 * emp:.1f}% | target tokens median {np.median(t):.0f} p90 {np.percentile(t, 90):.0f} | "
          f"single-sentence cut {100 * tr:.1f}%")

prompt <= 529 tokens + target <= 384 + EOS -> sequences <= 914 (context 1024)


## Batches, the adapted model, loss, generation
* **Training sequences** are `prompt + target + [EOS]`, right-padded; labels are −100 on the prompt and on padding. Micro-batches group sequences of similar length up to `MICRO_TOKENS` padded tokens; gradients are accumulated over `STEP_SEQS` sequences and the loss is the **mean over target tokens** of the step.
* **Logits only where a label exists**: the LM head (32k × 512) is applied to the target positions only, which saves most of the logit memory (prompts are ~60 % of every sequence).
* **Generation** is batched greedy decoding with a **KV cache** over left-padded prompts (position ids from the padding mask; the Phase-1 attention already handles left padding). Controls: no-repeat-n-gram (a token that would complete an n-gram already in this chunk's output is banned), repetition penalty on the tokens already generated for this chunk (CTRL-style: positive logits ÷ p, negative × p), `[EOS]` banned for the first `min_new_first` tokens of the document's first chunk.
* Cell 18 checks that batched, cached, left-padded generation produces **exactly** the same tokens as one-sequence, no-cache greedy decoding.

In [17]:
# Cell 16 — LoRMA (the Phase-3/4/5 code, unchanged) + model construction
_PI_IDX = {}
def _pi_index(d, dev):
    k = (d, str(dev))
    if k not in _PI_IDX:
        i = torch.arange(d, device=dev).view(d, 1); j = torch.arange(d, device=dev).view(1, d)
        _PI_IDX[k] = (j - i) % d
    return _PI_IDX[k]

class LoRMALinear(nn.Module):
    # h = I(B A) (W0 x).  'plus': y + (alpha/r) B (A y), B = 0 at init.  'pi': rotated rows of BA, identity at init.
    def __init__(self, base: nn.Linear, r: int, alpha: float, variant: str):
        super().__init__()
        assert variant in ("plus", "pi"), variant
        d = base.out_features
        assert r < d
        self.base, self.variant, self.d, self.scale = base, variant, d, alpha / r
        self.A = nn.Parameter(torch.empty(r, d))
        self.B = nn.Parameter(torch.empty(d, r))
        if variant == "plus":
            nn.init.kaiming_uniform_(self.A, a=math.sqrt(5)); nn.init.zeros_(self.B)
        else:
            nn.init.zeros_(self.A)
            nn.init.kaiming_uniform_(self.B, a=math.sqrt(5))
            with torch.no_grad():
                self.A[0, 0] = 1.0
                self.B[:, 0] = 1.0

    def forward(self, x):
        y = self.base(x)
        if self.variant == "plus":
            return y + (y @ self.A.t() @ self.B.t()) * self.scale
        M = self.B @ self.A
        Tm = torch.gather(M, 1, _pi_index(self.d, M.device))
        return y @ Tm.t()

def apply_adapter(module, targets, make):
    n = 0
    for name, child in list(module.named_children()):
        if isinstance(child, nn.Linear) and name in targets:
            setattr(module, name, make(child)); n += 1
        else:
            n += apply_adapter(child, targets, make)
    return n

def build_model():
    # Fresh copy of the Phase-1 weights; everything frozen except the LoRMA factors.
    model = copy.deepcopy(lm).cpu()
    for p in model.parameters(): p.requires_grad = False
    hp = HP[METHOD]
    ref = copy.deepcopy(model.backbone)
    n = apply_adapter(model.backbone, ADAPTER_TARGETS, lambda l: LoRMALinear(l, hp["r"], hp["alpha"], hp["variant"]))
    print(f"LoRMA[{hp['variant']}] on {n} Linear layers (r={hp['r']}, alpha={hp['alpha']})")
    _ids = torch.randint(5, VOCAB_SIZE, (1, 48))
    with torch.no_grad():
        a, _ = model.backbone.eval()(_ids); b, _ = ref.eval()(_ids)
    err = (a - b).abs().max().item()
    print(f"identity-at-init check: max|adapted - Phase-1| = {err:.2e}")
    assert err < 1e-4, "adapter does not start from the Phase-1 function"
    del ref
    model.to(device)
    tot = sum(p.numel() for p in model.parameters()); tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"params: total {tot/1e6:.2f}M | trainable {tr/1e6:.3f}M ({100*tr/tot:.2f}%)")
    model.n_total, model.n_trainable = tot, tr
    return model

def backbone_hidden(model, ids, mask=None, pos=None):
    # Backbone.forward without KV cache, with optional activation checkpointing.
    bb = model.backbone; N, T = ids.shape
    if pos is None: pos = torch.arange(T, device=ids.device).unsqueeze(0).expand(N, T)
    x = bb.emb_drop(bb.tok_emb(ids))
    for blk in bb.blocks:
        if model.training and GRAD_CKPT and torch.is_grad_enabled():
            x = checkpoint(lambda t, blk=blk: blk(t, rope=bb.rope, position_ids=pos, attention_mask=mask)[0], x, use_reentrant=False)
        else:
            x, _ = blk(x, rope=bb.rope, position_ids=pos, attention_mask=mask)
    return bb.norm_f(x)

In [18]:
# Cell 17 — Training pairs, micro-batches, loss
def make_pairs(docs, seed, empty_keep=EMPTY_KEEP):
    # -> list of (doc index, chunk index). Every chunk with a target + a random share of the empty ones (EOS-only target).
    rng = random.Random(seed); pairs = []
    for di, d in enumerate(docs):
        for j, t in enumerate(d["tgts"]):
            if t is not None or rng.random() < empty_keep: pairs.append((di, j))
    return pairs

def seq_of(docs, pair):
    di, j = pair; d = docs[di]
    p = prompt_ids(d, j)
    t = d["tgts"][j]
    tgt = ([] if t is None else t.astype(np.int64).tolist()) + ([] if (t is not None and d["trunc"][j]) else [EOS_ID])
    return p, tgt

def make_microbatches(docs, pairs, shuffle, seed=0, budget=None):
    budget = budget or MICRO_TOKENS
    lens = [len(prompt_ids(docs[di], j)) + (0 if docs[di]["tgts"][j] is None else len(docs[di]["tgts"][j])) + 1 for di, j in pairs]
    idx = list(range(len(pairs))); rng = random.Random(seed)
    if shuffle: rng.shuffle(idx)
    batches, pool = [], 2000
    for p in range(0, len(idx), pool):
        part = sorted(idx[p:p + pool], key=lambda i: lens[i])
        cur, longest = [], 0
        for i in part:
            if cur and (len(cur) + 1) * max(longest, lens[i]) > budget:
                batches.append(cur); cur, longest = [], 0
            cur.append(i); longest = max(longest, lens[i])
        if cur: batches.append(cur)
    if shuffle: rng.shuffle(batches)
    return [[pairs[i] for i in b] for b in batches]

def build_batch(docs, pairs):
    seqs, labs = [], []
    for pr in pairs:
        p, t = seq_of(docs, pr)
        full = p + t
        y = [-100] * (len(p) - 1) + t
        seqs.append(full[:-1]); labs.append(y)
    T = max(len(s) for s in seqs)
    x = np.full((len(seqs), T), PAD_ID, dtype=np.int64); y = np.full((len(seqs), T), -100, dtype=np.int64)
    for i, (s, l) in enumerate(zip(seqs, labs)):
        x[i, :len(s)] = s; y[i, :len(l)] = l
    return torch.from_numpy(x).to(device), torch.from_numpy(y).to(device)

def batch_loss(model, x, y):
    # sum of target-token NLL and the number of target tokens. No padding mask needed (right padding + causal attention).
    with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=USE_AMP):
        h = backbone_hidden(model, x)
        sel = y != -100
        logits = model.lm_head(h[sel])
    return F.cross_entropy(logits.float(), y[sel], reduction="sum"), int(sel.sum().item())

@torch.no_grad()
def eval_loss(model, docs, pairs):
    model.eval(); tot, n = 0.0, 0
    for b in make_microbatches(docs, pairs, shuffle=False, budget=2 * MICRO_TOKENS):
        x, y = build_batch(docs, b); l, k = batch_loss(model, x, y); tot += l.item(); n += k
    return tot / max(n, 1)

In [19]:
# Cell 18 — Batched greedy generation with KV cache, document summarisation, ROUGE
@torch.no_grad()
def generate(model, prompts, max_new, min_new=None, no_repeat_ngram=0, rep_penalty=1.0, amp=True):
    # prompts: list of token-id lists. Returns generated ids per prompt (without EOS).
    model.eval(); B = len(prompts); L = max(len(p) for p in prompts)
    ids = torch.full((B, L), PAD_ID, dtype=torch.long, device=device); mask = torch.zeros((B, L), dtype=torch.long, device=device)
    for i, p in enumerate(prompts):
        ids[i, L - len(p):] = torch.tensor(p, device=device); mask[i, L - len(p):] = 1
    pos = (mask.cumsum(1) - 1).clamp(min=0)
    min_new_t = torch.tensor(min_new if min_new is not None else [0] * B, device=device)
    with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=USE_AMP and amp):
        h, caches = model.backbone(ids, attention_mask=mask, position_ids=pos)
        logits = model.lm_head(h[:, -1]).float()
    out = [[] for _ in range(B)]; done = [False] * B; bans = [defaultdict(set) for _ in range(B)]
    seen = torch.zeros((B, logits.shape[-1]), dtype=torch.bool, device=device)
    cur = pos[:, -1].clone(); n = no_repeat_ngram
    for t in range(max_new):
        lg = logits
        if rep_penalty != 1.0:
            lg = torch.where(seen, torch.where(lg > 0, lg / rep_penalty, lg * rep_penalty), lg)
        lg[:, PAD_ID] = float("-inf"); lg[:, BOS_ID] = float("-inf")
        lg[:, EOS_ID] = torch.where(t < min_new_t, torch.full_like(lg[:, EOS_ID], float("-inf")), lg[:, EOS_ID])
        if n > 0 and t >= n - 1:
            rows, cols = [], []
            for b in range(B):
                if done[b]: continue
                for tok in bans[b].get(tuple(out[b][-(n - 1):]) if n > 1 else (), ()): rows.append(b); cols.append(tok)
            if rows: lg[rows, cols] = float("-inf")
        nxt = lg.argmax(-1)
        nl = nxt.tolist()
        for b in range(B):
            if done[b]: nl[b] = PAD_ID; continue
            if nl[b] == EOS_ID: done[b] = True; continue
            out[b].append(nl[b])
            if n > 0 and len(out[b]) >= n: bans[b][tuple(out[b][-n:-1]) if n > 1 else ()].add(out[b][-1])
        if all(done): break
        nxt = torch.tensor(nl, device=device)
        seen[torch.arange(B, device=device), nxt] = True
        mask = torch.cat([mask, torch.ones((B, 1), dtype=mask.dtype, device=device)], 1); cur = cur + 1
        with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=USE_AMP and amp):
            h, caches = model.backbone(nxt[:, None], attention_mask=mask, position_ids=cur[:, None], kv_caches=caches)
            logits = model.lm_head(h[:, -1]).float()
    return out

def select_chunks(n, k):
    # Head + tail + evenly spread middle (the audit's U-shaped source positions).
    if n <= k: return list(range(n))
    h = max(1, k // 4); t = max(1, k // 4); m = k - h - t
    mid = np.linspace(h, n - t - 1, m).round().astype(int).tolist() if m > 0 else []
    return sorted(set(list(range(h)) + mid + list(range(n - t, n))))

def predict_len_words(doc_words):
    return float(np.clip(LEN_MODEL["a"] * max(doc_words, 1) ** LEN_MODEL["b"], LEN_MODEL["lo"], LEN_MODEL["hi"]))

def _dedup(text):
    seen, out = set(), []
    for _, s in split_sentences(text):
        k = " ".join(words(s))
        if k and k in seen: continue
        seen.add(k); out.append(s)
    return " ".join(out)

LAST_GEN_STATS = {}
def summarize_docs(model, docs, gen=None, return_parts=False):
    g = dict(GEN, **(gen or {}))
    jobs = []                                                               # (doc index, chunk index)
    for di, d in enumerate(docs):
        for j in select_chunks(len(d["chunks"]), g["max_chunks"]): jobs.append((di, j))
    prompts = [prompt_ids(docs[di], j) for di, j in jobs]
    first = {di: min(j for dj, j in jobs if dj == di) for di in range(len(docs))}
    order = sorted(range(len(jobs)), key=lambda i: len(prompts[i]))
    outs = [None] * len(jobs); t0 = time.time()
    for s in range(0, len(order), g["batch"]):
        bi = order[s:s + g["batch"]]
        mins = [max(g["min_new_all"], g["min_new_first"] if jobs[i][1] == first[jobs[i][0]] else 0) for i in bi]
        res = generate(model, [prompts[i] for i in bi], g["max_new"], mins, g["no_repeat_ngram"], g["rep_penalty"])
        for i, r in zip(bi, res): outs[i] = r
    parts = defaultdict(list)
    for (di, j), o in zip(jobs, outs): parts[di].append((j, tokenizer.decode(o).strip()))
    LAST_GEN_STATS.update(chunks=len(jobs), empty_share=float(np.mean([len(o) == 0 for o in outs])),
                          tokens_per_chunk=float(np.mean([len(o) for o in outs])), hit_max_share=float(np.mean([len(o) >= g["max_new"] for o in outs])))
    summaries = []
    for di, d in enumerate(docs):
        txt = " ".join(t for _, t in sorted(parts[di]) if t)
        if g["dedup"]: txt = _dedup(txt)
        budget = int(round(g["len_factor"] * predict_len_words(d["words"])))
        w = txt.split()
        summaries.append(" ".join(w[:budget]))
    secs = time.time() - t0
    return (summaries, secs, parts) if return_parts else (summaries, secs)

# ---- ROUGE: fast exact implementation verified against rouge_score (as in the audit, Cell 9) ----
try:
    from rouge_score import rouge_scorer
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rouge-score"], check=True)
    from rouge_score import rouge_scorer
_RS = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
def rtok(text): return _RS._tokenizer.tokenize(text)
def lcs_len(a, b):
    if not a or not b: return 0
    masks = {}
    for i, x in enumerate(a): masks[x] = masks.get(x, 0) | (1 << i)
    full = (1 << len(a)) - 1; V = full
    for y in b:
        U = V & masks.get(y, 0)
        V = ((V + U) | (V - U)) & full
    return len(a) - bin(V).count("1")
def _f(o, nc, nr):
    p, r = (o / nc if nc else 0.0), (o / nr if nr else 0.0)
    return p, r, (2 * p * r / (p + r) if p + r else 0.0)
def rouge(cand, ref):
    c, r = rtok(cand), rtok(ref); out = {}
    for n in (1, 2):
        cc, rc = Counter(zip(*[c[i:] for i in range(n)])), Counter(zip(*[r[i:] for i in range(n)]))
        out[f"r{n}"] = _f(sum(min(v, rc[k]) for k, v in cc.items()), sum(cc.values()), sum(rc.values()))
    out["rL"] = _f(lcs_len(r, c), len(c), len(r))
    return out
_c, _r = "the appellant was convicted under section 302 by the court", "the appellant was convicted by the sessions court under section 302"
assert all(abs(_RS.score(_r, _c)[k].fmeasure - rouge(_c, _r)[k2][2]) < 1e-9 for k, k2 in (("rouge1", "r1"), ("rouge2", "r2"), ("rougeL", "rL")))

def rouge_eval(cands, refs):
    R = defaultdict(list)
    for c, r in zip(cands, refs):
        for k, v in rouge(c, r).items(): R[k].append(v)
    return {k: dict(p=float(np.mean([x[0] for x in v])), r=float(np.mean([x[1] for x in v])), f=float(np.mean([x[2] for x in v])))
            for k, v in R.items()}, {k: np.asarray([x[2] for x in v]) for k, v in R.items()}

In [20]:
# Cell 19 — Self-test: batched + cached + left-padded greedy == one-sequence, no-cache greedy (on the Phase-1 model)
def _greedy_ref(model, p, steps):
    seq = list(p); out = []
    for _ in range(steps):
        with torch.no_grad():
            h, _ = model.backbone(torch.tensor([seq], device=device))
            lg = model.lm_head(h[:, -1]).float()[0]
        lg[PAD_ID] = float("-inf"); lg[BOS_ID] = float("-inf")
        t = int(lg.argmax())
        if t == EOS_ID: break
        out.append(t); seq.append(t)
    return out

_tf32 = torch.backends.cuda.matmul.allow_tf32
torch.backends.cuda.matmul.allow_tf32 = False
_long = [BOS_ID] + tokenizer.encode(VAL_ALL[0]["doc"][:2400])[:420] + SEP_IDS      # > local window (256): the band mask binds
_tp = [[BOS_ID] + tokenizer.encode("The appellant was convicted by the Sessions Court and"), [BOS_ID] + tokenizer.encode("Held:"), _long]
_steps = 24
_b = generate(lm, _tp, _steps, amp=False)
for p, g in zip(_tp, _b):
    r = _greedy_ref(lm, p, _steps)
    assert g == r, f"cached batched generation differs from the reference:\n{g}\n{r}"
torch.backends.cuda.matmul.allow_tf32 = _tf32
print(f"generation self-test OK: batched + KV cache + left padding == reference greedy (prompts of {[len(p) for p in _tp]} tokens)")
print("Phase-1 continuation (no fine-tuning):", repr(tokenizer.decode(_b[0])[:200]))

generation self-test OK: batched + KV cache + left padding == reference greedy (prompts of [10, 4, 426] tokens)
Phase-1 continuation (no fine-tuning): ' sentenced to undergo rigorous imprisonment for life. The appellant was sentenced to undergo rigorous imprisonment for life. The appellant was sentenced to'


## Build the data and train

In [21]:
# Cell 20 — Prepare data, build the model, train
seed_everything(SEED)
if SMOKE_TEST:
    _r = random.Random(SEED)
    train_docs = _r.sample(TR_ALL, min(SMOKE_TRAIN_DOCS, len(TR_ALL))); val_docs = VAL_ALL[:SMOKE_VAL_DOCS]
else:
    train_docs, val_docs = TR_ALL, VAL_ALL
_t0 = time.time()
prepare_docs(train_docs, "tr_smoke" if SMOKE_TEST else "tr", True)
prepare_docs(val_docs, "val_smoke" if SMOKE_TEST else "val", True)
print(f"preparation {time.time() - _t0:.0f}s")
pair_stats(train_docs, "TR"); pair_stats(val_docs, "VAL")
val_pairs = make_pairs(val_docs, seed=0, empty_keep=EMPTY_KEEP)

def param_groups(model):
    hp = HP[METHOD]
    ps = [p for p in model.parameters() if p.requires_grad]
    return [dict(params=ps, lr=hp["lr"], base_lr=hp["lr"], weight_decay=hp["wd"])]

def quick_rouge(model, docs, n):
    if not n: return None
    s, _ = summarize_docs(model, docs[:n])
    m, _ = rouge_eval(s, [d["summ"] for d in docs[:n]])
    return m

def train_model(model, epochs, max_steps=None):
    opt = torch.optim.AdamW(param_groups(model), betas=(0.9, 0.999))
    scaler = torch.amp.GradScaler("cuda", enabled=USE_SCALER)
    trainables = [p for p in model.parameters() if p.requires_grad]
    n_pairs = len(make_pairs(train_docs, seed=SEED + 1))
    steps_per_epoch = math.ceil(n_pairs / STEP_SEQS); total = epochs * steps_per_epoch
    if max_steps: total = min(total, max_steps)
    warm = max(1, int(WARMUP_FRAC * total))
    def lr_mult(s):
        if s < warm: return (s + 1) / warm
        return 0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * min(1.0, (s - warm) / max(1, total - warm))))
    print(f"~{n_pairs} training sequences/epoch | ~{steps_per_epoch} steps/epoch ({STEP_SEQS} seqs/step) | total {total} steps")
    if device.type == "cuda": torch.cuda.reset_peak_memory_stats()
    history, losses, step, done = [], [], 0, False
    best = dict(score=None, epoch=0, state=None); t_train = time.time(); tok_seen = 0
    v0 = eval_loss(model, val_docs, val_pairs); print(f"VAL loss before training (Phase-1 model, target tokens): {v0:.4f}")
    for ep in range(1, epochs + 1):
        pairs = make_pairs(train_docs, seed=SEED + ep)
        batches = make_microbatches(train_docs, pairs, shuffle=True, seed=SEED + ep)
        model.train(); t0 = time.time(); ep_sum, ep_tok, acc, acc_tok, acc_loss = 0.0, 0, 0, 0, None
        for bi, b in enumerate(batches):
            x, y = build_batch(train_docs, b)
            l, k = batch_loss(model, x, y)
            # normalise by the target tokens of the whole step: sum-losses are divided by an estimate (k * STEP_SEQS / len(b))
            scaler.scale(l / max(1.0, k * STEP_SEQS / len(b))).backward()
            ep_sum += l.item(); ep_tok += k; tok_seen += x.numel(); acc += len(b)
            losses.append(l.item() / max(k, 1))
            if acc >= STEP_SEQS or bi == len(batches) - 1:
                m = lr_mult(step)
                for g in opt.param_groups: g["lr"] = g["base_lr"] * m
                scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(trainables, GRAD_CLIP)
                scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); step += 1; acc = 0
                if step == 20:
                    per = (time.time() - t0) / 20
                    print(f"  >> speed: {per:.2f} s/step | projected {per * steps_per_epoch / 60:.1f} min/epoch "
                          f"({per * steps_per_epoch * epochs / 60:.0f} min for {epochs} epochs, plus evaluation)"
                          + (f" | peak GPU {torch.cuda.max_memory_allocated() / 2**30:.1f} GB" if device.type == "cuda" else ""))
                if step % 100 == 0:
                    print(f"  ep {ep} step {step}/{total} | loss {np.mean(losses[-300:]):.4f} | lr x{m:.3f} | {time.time() - t0:.0f}s")
                if max_steps and step >= max_steps: done = True; break
                if TIME_BUDGET_MIN and (time.time() - t_train) / 60 > TIME_BUDGET_MIN:
                    print(f"  TIME_BUDGET_MIN={TIME_BUDGET_MIN} reached -> stopping"); done = True; break
        vl = eval_loss(model, val_docs, val_pairs)
        qr = quick_rouge(model, val_docs, min(VAL_GEN_DOCS_EPOCH, len(val_docs)))
        rec = dict(epoch=ep, train_loss=ep_sum / max(ep_tok, 1), val_loss=vl, seconds=time.time() - t0, steps=step)
        if qr: rec.update(q_r1=qr["r1"]["f"], q_r2=qr["r2"]["f"], q_rL=qr["rL"]["f"])
        score = -vl if SELECT_BY == "loss" else (qr["r1"]["f"] + qr["rL"]["f"]) / 2
        if best["score"] is None or score > best["score"]:
            best = dict(score=score, epoch=ep, state={n: p.detach().cpu().clone() for n, p in model.named_parameters() if p.requires_grad})
            torch.save(best, RUN_DIR / "best_trainable.pt")
        print(f"epoch {ep}: " + " | ".join(f"{k} {v:.4f}" if isinstance(v, float) else f"{k} {v}" for k, v in rec.items() if k != "epoch"))
        history.append(rec)
        if done: break
    stats = dict(train_seconds=time.time() - t_train, steps=step, tokens_seen=tok_seen, val_loss_init=v0,
                 peak_gpu_gb=(torch.cuda.max_memory_allocated() / 2**30) if device.type == "cuda" else 0.0,
                 first_losses=float(np.mean(losses[:20])) if losses else None, last_losses=float(np.mean(losses[-20:])) if losses else None)
    return history, best, stats

model = build_model()
history, best, stats = train_model(model, EPOCHS, max_steps=SMOKE_STEPS if SMOKE_TEST else None)
model.load_state_dict(best["state"], strict=False)
print(f"\nbest epoch {best['epoch']} (by {SELECT_BY}) | train {stats['train_seconds']/60:.1f} min | peak GPU {stats['peak_gpu_gb']:.1f} GB")

prepare tr:   0%|          | 0/6759 [00:00<?, ?it/s]

prepare val:   0%|          | 0/200 [00:00<?, ?it/s]

preparation 125s
TR: 6759 docs | chunks/doc mean 10.2 median 8 | chunk pairs with a target 47133 | empty chunks 31.8% | target tokens median 115 p90 277 | single-sentence cut 0.1%
VAL: 200 docs | chunks/doc mean 11.0 median 9 | chunk pairs with a target 1525 | empty chunks 30.6% | target tokens median 110 p90 279 | single-sentence cut 0.1%
LoRMA[plus] on 84 Linear layers (r=16, alpha=32)
identity-at-init check: max|adapted - Phase-1| = 0.00e+00
params: total 56.90M | trainable 1.966M (3.46%)
~58259 training sequences/epoch | ~1821 steps/epoch (32 seqs/step) | total 7284 steps
VAL loss before training (Phase-1 model, target tokens): 2.7140
  >> speed: 1.18 s/step | projected 36.0 min/epoch (144 min for 4 epochs, plus evaluation) | peak GPU 6.2 GB
  ep 1 step 100/7284 | loss 2.8594 | lr x0.459 | 112s
  ep 1 step 200/7284 | loss 1.8865 | lr x0.917 | 220s
  ep 1 step 300/7284 | loss 1.8593 | lr x1.000 | 329s
  ep 1 step 400/7284 | loss 1.8059 | lr x0.999 | 435s
  ep 1 step 500/7284 | loss 

In [22]:
# Cell 21 — Smoke-test verdict + sample outputs (always printed)
val_extra = {}
_n = min(2, len(val_docs))
_s, _, _parts = summarize_docs(model, val_docs[:_n], return_parts=True)
for d, s in zip(val_docs[:_n], _s):
    print(f"--- {d['id']} | {len(d['chunks'])} chunks | generated {len(s.split())} words | reference {len(d['summ'].split())} words")
    print("GENERATED:", s[:700].replace("\n", " "))
    print("REFERENCE:", d["summ"][:700].replace("\n", " "))
if SMOKE_TEST:
    drop = stats["last_losses"] / stats["first_losses"]
    _ss = summarize_docs(model, val_docs)[0]
    m, _ = rouge_eval(_ss, [d["summ"] for d in val_docs])
    val_extra.update(smoke_r1=m["r1"]["f"], smoke_r1_p=m["r1"]["p"], smoke_r1_r=m["r1"]["r"], smoke_rL=m["rL"]["f"],
                     smoke_words_gen=float(np.mean([len(x.split()) for x in _ss])), smoke_words_ref=float(np.mean([len(d["summ"].split()) for d in val_docs])),
                     smoke_gen_stats=dict(LAST_GEN_STATS))
    print(f"generated {val_extra['smoke_words_gen']:.0f} words vs reference {val_extra['smoke_words_ref']:.0f} | R1 P {m['r1']['p']:.3f} R {m['r1']['r']:.3f} | "
          f"chunks with an empty output {LAST_GEN_STATS['empty_share']:.2f} | tokens/chunk {LAST_GEN_STATS['tokens_per_chunk']:.0f}")
    print(f"\nloss {stats['first_losses']:.3f} -> {stats['last_losses']:.3f} (x{drop:.2f}) | VAL loss {history[-1]['val_loss']:.3f} "
          f"(Phase-1 {stats['val_loss_init']:.3f}) | R1 {m['r1']['f']:.3f} RL {m['rL']['f']:.3f} on {len(val_docs)} docs")
    print("SMOKE VERDICT:", "OK -> run the full job" if (drop < 0.95 and history[-1]["val_loss"] < stats["val_loss_init"]) else
          "NOT OK: loss did not fall / VAL loss did not improve -> paste the log to Claude")
    print("Check the '>> speed' line: it projects minutes per epoch; set EPOCHS / TIME_BUDGET_MIN accordingly.")

--- 2459 | 12 chunks | generated 28 words | reference 912 words
GENERATED: The first respondent, a candidate in the Indian Legislative Assembly, was declared elected to the State Legislative Assembly. The first respondent was also charged with other corrupt practices.
REFERENCE: The election of the first respondent to the Andhra Pradesh Legislative Assembly at the General Election hold in 1967 was challenged by the appellants in an election petition. Various corrupt practices were alleged appellants to have been committed by the first respondent, his agents and supporters. One of these was that the first respondent paid a bribe to one R who had also filed nomination papers, in order to secure his withdrawal as a candidate. After the issues were framed the appellants made an application to the Court for impleading R but it was dismissed. The first respondent then filed an application under section 86(1) praying for the dismissal of the election petition on the ground
--- 2718 | 6 chunks

## Decoding sweep and VAL evaluation (full runs)
The decoding configuration is chosen on the first `SWEEP_DOCS` VAL documents by (R1 + RL)/2, then **all 200** VAL documents are summarised with it (the audit's baselines were scored on the same 200). The analysis covers what the PDF asks for: scores vs the extractive baselines, length vs score, redundancy (repeated n-grams), abstractiveness of the outputs (novel n-grams vs the document), scores by document length, and cost.

In [23]:
# Cell 22 — Decoding sweep (on VAL[:SWEEP_DOCS]) -> best GEN
sweep_res = []
if not SMOKE_TEST and SWEEP_DOCS and DECODE_SWEEP:
    _docs = val_docs[:SWEEP_DOCS]; _refs = [d["summ"] for d in _docs]
    for cfg in DECODE_SWEEP:
        s, secs = summarize_docs(model, _docs, cfg); m, _ = rouge_eval(s, _refs)
        wg = float(np.mean([len(x.split()) for x in s]))
        sweep_res.append(dict(cfg=cfg, r1=m["r1"]["f"], r2=m["r2"]["f"], rL=m["rL"]["f"], r1_p=m["r1"]["p"], r1_r=m["r1"]["r"], words=wg,
                              empty_chunks=LAST_GEN_STATS["empty_share"], secs=secs))
        print(f"  {cfg}: R1 {m['r1']['f']:.4f} (P {m['r1']['p']:.3f} R {m['r1']['r']:.3f}) R2 {m['r2']['f']:.4f} RL {m['rL']['f']:.4f} | "
              f"words {wg:.0f} | empty chunks {LAST_GEN_STATS['empty_share']:.2f} | {secs:.0f}s")
    bestcfg = max(sweep_res, key=lambda r: (r["r1"] + r["rL"]) / 2)["cfg"]
    GEN.update(bestcfg); print("chosen decoding:", bestcfg)
else:
    print("decoding sweep skipped")

  {'no_repeat_ngram': 4, 'rep_penalty': 1.1, 'min_new_all': 0}: R1 0.1996 (P 0.805 R 0.126) R2 0.1094 RL 0.1391 | words 94 | empty chunks 0.79 | 42s
  {'no_repeat_ngram': 4, 'rep_penalty': 1.1, 'min_new_all': 32}: R1 0.5306 (P 0.718 R 0.456) R2 0.2853 RL 0.2791 | words 533 | empty chunks 0.00 | 72s
  {'no_repeat_ngram': 4, 'rep_penalty': 1.1, 'min_new_all': 64}: R1 0.5864 (P 0.654 R 0.577) R2 0.3210 RL 0.2982 | words 769 | empty chunks 0.00 | 74s
  {'no_repeat_ngram': 3, 'rep_penalty': 1.0, 'min_new_all': 32}: R1 0.5335 (P 0.707 R 0.463) R2 0.2867 RL 0.2836 | words 549 | empty chunks 0.00 | 86s
  {'no_repeat_ngram': 0, 'rep_penalty': 1.2, 'min_new_all': 32}: R1 0.5162 (P 0.699 R 0.452) R2 0.2718 RL 0.2684 | words 549 | empty chunks 0.00 | 94s
chosen decoding: {'no_repeat_ngram': 4, 'rep_penalty': 1.1, 'min_new_all': 64}


In [24]:
# Cell 23 — VAL evaluation: ROUGE (+ BERTScore), lengths, redundancy, abstractiveness, by document length
val_res = {}
if not SMOKE_TEST:
    refs = [d["summ"] for d in val_docs]
    val_sum, t_val = summarize_docs(model, val_docs)
    val_extra["gen_stats"] = dict(LAST_GEN_STATS)
    m, per = rouge_eval(val_sum, refs)
    val_res = {k: v["f"] for k, v in m.items()}
    val_extra.update(seconds=t_val, sec_per_doc=t_val / len(val_docs),
                     words_gen=float(np.mean([len(s.split()) for s in val_sum])), words_ref=float(np.mean([len(r.split()) for r in refs])),
                     r1_p=m["r1"]["p"], r1_r=m["r1"]["r"])
    if RUN_BERTSCORE and device.type == "cuda":
        try:
            import bert_score
        except ImportError:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "bert-score"], check=True); import bert_score
        import transformers; transformers.logging.set_verbosity_error()
        _bs = bert_score.BERTScorer(model_type=BERTSCORE_MODEL, lang="en", device="cuda", batch_size=8)
        P_, R_, F_ = _bs.score(val_sum, refs)
        val_res.update(bs_p=float(P_.mean()), bs_r=float(R_.mean()), bs_f=float(F_.mean())); per["bs"] = F_.numpy()
        del _bs; gc.collect(); torch.cuda.empty_cache()
    print(f"===== VAL ({len(val_docs)} docs) | R1 {val_res['r1']:.4f} (P {m['r1']['p']:.3f} R {m['r1']['r']:.3f}) | R2 {val_res['r2']:.4f} | "
          f"RL {val_res['rL']:.4f}" + (f" | BERTScore F1 {val_res['bs_f']:.4f}" if "bs_f" in val_res else ""))
    for name, b in BASELINES.items():
        print(f"      vs {name:6s}: R1 {val_res['r1'] - b['r1']:+.4f} | R2 {val_res['r2'] - b['r2']:+.4f} | RL {val_res['rL'] - b['rL']:+.4f}"
              + (f" | BERTScore {val_res['bs_f'] - b['bs']:+.4f}" if "bs_f" in val_res else ""))
    print(f"      words: generated {val_extra['words_gen']:.0f} vs reference {val_extra['words_ref']:.0f} | {val_extra['sec_per_doc']:.1f} s/doc")

    # redundancy: share of repeated 3-grams inside each output; abstractiveness: novel n-grams w.r.t. the document
    def rep3(t):
        w = words(t); g = list(zip(w, w[1:], w[2:]))
        return 1 - len(set(g)) / max(len(g), 1)
    def novel(t, d, n):
        a, s = words(d), words(t); ga = set(zip(*[a[i:] for i in range(n)])); gs = list(zip(*[s[i:] for i in range(n)]))
        return sum(x not in ga for x in gs) / max(len(gs), 1)
    val_extra["rep3_gen"] = float(np.mean([rep3(s) for s in val_sum])); val_extra["rep3_ref"] = float(np.mean([rep3(r) for r in refs]))
    val_extra["novel"] = {n: float(np.mean([novel(s, d["doc"], n) for s, d in zip(val_sum, val_docs)])) for n in (1, 2, 4)}
    print(f"      redundancy (repeated 3-grams): generated {val_extra['rep3_gen']:.3f} vs reference {val_extra['rep3_ref']:.3f}")
    print(f"      novel n-grams vs the document (1/2/4): {[round(v, 3) for v in val_extra['novel'].values()]}  (references: 0.044 / 0.223 / 0.484)")

    ntok_v = np.asarray([d["ntok"] for d in val_docs]); cuts = np.percentile(ntok_v, [33.3, 66.7]); bk = np.digitize(ntok_v, cuts); rows = []
    print("      by document length (token terciles):")
    for b_, name in enumerate(("short", "medium", "long")):
        sel = bk == b_
        r_ = dict(bucket=name, n=int(sel.sum()), median_tokens=float(np.median(ntok_v[sel])), r1=float(per["r1"][sel].mean()),
                  rL=float(per["rL"][sel].mean()), truncated=float(np.mean([len(val_docs[i]["chunks"]) > GEN["max_chunks"] for i in np.where(sel)[0]])))
        if "bs" in per: r_["bs"] = float(per["bs"][sel].mean())
        rows.append(r_)
        print(f"        {name:6s}: {sel.sum():3d} docs | median {r_['median_tokens']:6.0f} tokens | R1 {r_['r1']:.3f} RL {r_['rL']:.3f}"
              + (f" BS {r_['bs']:.3f}" if "bs" in r_ else "") + f" | chunks skipped in {100 * r_['truncated']:.0f}% of docs")
    val_extra["by_length"] = rows
    (RUN_DIR / "val_summaries.json").write_text(json.dumps([dict(id=d["id"], summary=s) for d, s in zip(val_docs, val_sum)], ensure_ascii=False))
else:
    print("no VAL evaluation (smoke run)")

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.42GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

===== VAL (200 docs) | R1 0.5686 (P 0.631 R 0.574) | R2 0.3043 | RL 0.2868 | BERTScore F1 0.8513
      vs tfidf : R1 -0.0063 | R2 -0.0143 | RL -0.0140 | BERTScore -0.0020
      vs lead  : R1 +0.0464 | R2 +0.0535 | RL +0.0431 | BERTScore +0.0192
      vs oracle: R1 -0.2174 | R2 -0.2999 | RL -0.2473 | BERTScore -0.0342
      words: generated 725 vs reference 848 | 1.1 s/doc
      redundancy (repeated 3-grams): generated 0.116 vs reference 0.099
      novel n-grams vs the document (1/2/4): [0.054, 0.219, 0.48]  (references: 0.044 / 0.223 / 0.484)
      by document length (token terciles):
        short :  67 docs | median   2033 tokens | R1 0.558 RL 0.310 BS 0.860 | chunks skipped in 0% of docs
        medium:  66 docs | median   4260 tokens | R1 0.574 RL 0.290 BS 0.851 | chunks skipped in 0% of docs
        long  :  67 docs | median   8362 tokens | R1 0.574 RL 0.260 BS 0.844 | chunks skipped in 54% of docs


In [25]:
# Cell 24 — Test summaries -> prediction.json (list format) -> submission.zip (full runs only)
def verify_submission(zip_path, docs):
    with zipfile.ZipFile(zip_path) as z:
        names = z.namelist()
        assert names == ["prediction.json"], f"the zip must contain ONLY prediction.json at its root, found {names}"
        sub = json.loads(z.read("prediction.json"))
    assert isinstance(sub, list), "prediction.json must be a JSON list of {id, summary}"
    want = [str(d["id"]) for d in docs]; got = [str(e["id"]) for e in sub]
    assert len(got) == len(set(got)), "duplicate ids"
    assert set(got) == set(want), f"id mismatch: missing {len(set(want) - set(got))}, extra {len(set(got) - set(want))}"
    for e in sub:
        assert set(e) == {"id", "summary"}, f"unexpected keys {set(e)}"
        assert isinstance(e["summary"], str) and e["summary"].strip(), f"{e['id']}: empty summary"
    print(f"verified: {len(sub)} summaries, one non-empty string per test id")

if not SMOKE_TEST:
    prepare_docs(TEST_ALL, "test", False)
    test_sum, t_test = summarize_docs(model, TEST_ALL)
    sub = [dict(id=d["id"], summary=s) for d, s in zip(TEST_ALL, test_sum)]          # id keeps its type from test.json
    out_dir = RUN_DIR / "submission"; out_dir.mkdir(parents=True, exist_ok=True)
    (out_dir / "prediction.json").write_text(json.dumps(sub, ensure_ascii=False))
    with zipfile.ZipFile(out_dir / "submission.zip", "w", zipfile.ZIP_DEFLATED) as z:
        z.write(out_dir / "prediction.json", arcname="prediction.json")
    verify_submission(out_dir / "submission.zip", TEST_ALL)
    print(f"test: {len(sub)} docs in {t_test:.0f}s | words/summary mean {np.mean([len(s.split()) for s in test_sum]):.0f}")
else:
    print("smoke run: no submission written")

prepare test:   0%|          | 0/100 [00:00<?, ?it/s]

verified: 100 summaries, one non-empty string per test id
test: 100 docs in 120s | words/summary mean 693


In [26]:
# Cell 25 — Save trainable weights + run log
def _jsonable(o):
    if isinstance(o, dict): return {str(k): _jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [_jsonable(v) for v in o]
    if isinstance(o, np.ndarray): return o.tolist()
    if isinstance(o, (np.floating, np.integer, np.bool_)): return o.item()
    if isinstance(o, Path): return str(o)
    return o
log = dict(run=RUN_NAME, method=METHOD, hp=HP[METHOD], chunk_tokens=CHUNK_TOKENS, tgt_max=TGT_MAX, empty_keep=EMPTY_KEEP,
           min_align_f1=MIN_ALIGN_F1, epochs=EPOCHS, micro_tokens=MICRO_TOKENS, step_seqs=STEP_SEQS, select_by=SELECT_BY,
           gen=GEN, decode_sweep=sweep_res, params_total=model.n_total, params_trainable=model.n_trainable,
           history=history, best_epoch=best["epoch"], stats=stats, val=val_res, val_extra=val_extra, baselines=BASELINES,
           n_train_docs=len(train_docs), n_val_docs=len(val_docs), smoke=SMOKE_TEST, phase1_local_window=lm_cfg.local_window)
(RUN_DIR / "run_log.json").write_text(json.dumps(_jsonable(log), indent=2))
torch.save({"trainable": best["state"], "method": METHOD, "hp": HP[METHOD], "gen": GEN, "log": _jsonable(log)}, RUN_DIR / f"summ_{RUN_NAME}.pt")
print("saved:", sorted(p.name for p in RUN_DIR.iterdir()))

saved: ['best_trainable.pt', 'run_log.json', 'submission', 'summ_p6_dec02_lorma.pt', 'val_summaries.json']


## KD teacher export (for run #3)
Two signals, both keyed by `"<doc uid>|<chunk index>"` so the student notebook (same tokenizer, same chunking and alignment code) can match them exactly:
* **Word-level KD** (Hinton et al. 2015; Kim & Rush 2016): for every TR chunk pair with a target, the teacher's **top-k logits** at each gold target position (+ the log-sum-exp over the full vocabulary, so the student can recover true probabilities of the top-k tokens). Possible across architectures only because all Phase-1 models share one tokenizer.
* **Sequence-level KD** (Kim & Rush 2016): the teacher's own greedy summaries for up to `TEACHER_GEN_PAIRS` TR chunks (with the chosen decoding config). The student trains on them as targets.

In [27]:
# Cell 26 — Export teacher signals
if EXPORT_TEACHER and not SMOKE_TEST:
    t0 = time.time(); model.eval()
    tr_pairs = [(di, j) for di, d in enumerate(train_docs) for j, t in enumerate(d["tgts"]) if t is not None]
    topk = {}
    with torch.no_grad():
        for b in tqdm(make_microbatches(train_docs, tr_pairs, shuffle=False, budget=2 * MICRO_TOKENS), desc="teacher top-k", leave=False):
            x, y = build_batch(train_docs, b)
            with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=USE_AMP):
                h = backbone_hidden(model, x)
            for r, (di, j) in enumerate(b):
                sel = y[r] != -100
                lg = model.lm_head(h[r][sel].float()).float()
                v, ix = lg.topk(TEACHER_TOPK, dim=-1); lse = torch.logsumexp(lg, -1)
                topk[f"{train_docs[di]['uid']}|{j}"] = (v.half().cpu().numpy(), ix.to(torch.int32).cpu().numpy().astype(np.uint16),
                                                        lse.cpu().numpy().astype(np.float32))
    print(f"top-k logits for {len(topk)} pairs in {time.time() - t0:.0f}s")
    t1 = time.time(); gen_out = {}
    rng = random.Random(SEED); cand = list(tr_pairs); rng.shuffle(cand); cand = cand[:TEACHER_GEN_PAIRS]
    cand.sort(key=lambda p: len(prompt_ids(train_docs[p[0]], p[1])))
    for s in range(0, len(cand), GEN["batch"]):
        b = cand[s:s + GEN["batch"]]
        res = generate(model, [prompt_ids(train_docs[di], j) for di, j in b], GEN["max_new"], None, GEN["no_repeat_ngram"], GEN["rep_penalty"])
        for (di, j), o in zip(b, res): gen_out[f"{train_docs[di]['uid']}|{j}"] = np.asarray(o, dtype=np.uint16)
        if (time.time() - t1) / 60 > TEACHER_GEN_MIN:
            print(f"TEACHER_GEN_MIN reached after {len(gen_out)} chunks"); break
    print(f"teacher generations for {len(gen_out)} chunks in {time.time() - t1:.0f}s")
    meta = dict(teacher=RUN_NAME, method=METHOD, hp=HP[METHOD], chunk_tokens=CHUNK_TOKENS, tgt_max=TGT_MAX, min_align_f1=MIN_ALIGN_F1,
                max_full_chars=MAX_FULL_CHARS, tag_template=TAG_TEMPLATE, sep_text=SEP_TEXT, topk=TEACHER_TOPK, gen=GEN,
                params_total=int(model.n_total), params_trainable=int(model.n_trainable), val=val_res, val_checksum=VAL_CHECKSUM_EXPECTED)
    path = RUN_DIR / f"teacher_{RUN_NAME}.pt"
    torch.save(dict(meta=meta, topk=topk, gen=gen_out), path)
    print(f"saved {path} ({path.stat().st_size / 1e6:.0f} MB) | total {time.time() - t0:.0f}s")
else:
    print("teacher export skipped")

teacher top-k:   0%|          | 0/1280 [00:00<?, ?it/s]

top-k logits for 47133 pairs in 620s
teacher generations for 15000 chunks in 893s
saved /kaggle/working/runs/p6_dec02_lorma/teacher_p6_dec02_lorma.pt (310 MB) | total 1518s


In [28]:
# Cell 27 — PASTE TO CLAUDE
short = dict(run=RUN_NAME, smoke=SMOKE_TEST, hp=HP[METHOD], params_trainable=model.n_trainable, params_total=model.n_total,
             history=history, best_epoch=best["epoch"], stats={k: (round(v, 4) if isinstance(v, float) else v) for k, v in stats.items()},
             gen=GEN, decode_sweep=sweep_res, val={k: round(v, 4) for k, v in val_res.items()},
             val_extra={k: v for k, v in val_extra.items() if k != "seconds"})
print("================ PASTE TO CLAUDE ================")
print(json.dumps(_jsonable(short), separators=(",", ":")))
print("=================================================")

================ PASTE TO CLAUDE ================
{"run":"p6_dec02_lorma","smoke":false,"hp":{"lr":0.0006,"wd":0.0,"r":16,"alpha":32,"variant":"plus"},"params_trainable":1966080,"params_total":56898048,"history":[{"epoch":1,"train_loss":1.8048929143300876,"val_loss":1.674443431770259,"seconds":1607.6141374111176,"steps":1461,"q_r1":0.18537914587835103,"q_r2":0.10182196245674753,"q_rL":0.13301950758588577},{"epoch":2,"train_loss":1.6284063100079407,"val_loss":1.6223612450653864,"seconds":1612.1156198978424,"steps":2920,"q_r1":0.1940114219300999,"q_r2":0.10696361347293952,"q_rL":0.1399708728070671},{"epoch":3,"train_loss":1.5560063106644557,"val_loss":1.613291876244523,"seconds":1614.9316427707672,"steps":4380,"q_r1":0.13935237529976158,"q_r2":0.07275451789681052,"q_rL":0.10446292759487785},{"epoch":4,"train_loss":1.4981201819918002,"val_loss":1.5880760134628982,"seconds":1616.5078899860382,"steps":5841,"q_r1":0.2099529293163469,"q_r2":0.11607338907522621,"q_rL":0.14707811802974421}],"be